# Neural Models Lab — XOR, Activations, and Output Layers

**Shrihari Karthik (2023B4A70868G)** — Lab Exercise: *Neural Models: Learning, Depth, Activations, and Output Layers*

Scenario: two redundant binary sensors `x1, x2`; raise a **disagreement warning** exactly when
exactly one sensor is active (XOR). Part 2 extends this to a three-class decision.

Everything in this notebook is reproducible: one global seed, full-batch training, CPU only.
Run top to bottom; the last cell prints the exact numbers quoted in `report.pdf`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

SEED = 9            # see report §4A: seed 0 left tanh/ReLU in a 3/4 local minimum
STEPS = 4000
LR = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_printoptions(precision=6, sci_mode=False)
print("torch", torch.__version__)

## Task 1 — Problem specification (dataset)

- Input space  $\mathcal{X} = \{0,1\}^2$ (4 points)
- Output space $\mathcal{Y} = \{0,1\}$, interpreted through $p = \sigma(z) \in (0,1)$
- Examples: (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->0

In [ ]:
X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])     # column vector, float -> required by BCEWithLogitsLoss

print("X =\n", X)
print("y =", Y.view(-1).tolist())
assert X.shape == (4, 2) and Y.shape == (4, 1)

### Linear-separability check (empirical support for the Task 1 prediction)

A single affine map + sigmoid is a *linear* classifier. Trained to convergence it cannot
get all four XOR points right. The cell below trains one and shows it plateaus at
loss $\approx \ln 2 = 0.6931$ with all four probabilities $\approx 0.5$.

In [ ]:
torch.manual_seed(SEED)
linear_only = nn.Linear(2, 1)                       # affine, then sigmoid via the loss
opt = torch.optim.Adam(linear_only.parameters(), lr=LR)
lossf = nn.BCEWithLogitsLoss()

for step in range(STEPS):
    opt.zero_grad()
    loss = lossf(linear_only(X), Y)
    loss.backward()
    opt.step()

with torch.no_grad():
    p_lin = torch.sigmoid(linear_only(X)).view(-1)
print(f"linear model  final loss = {loss.item():.6f}   (ln 2 = {np.log(2):.6f})")
print("probabilities :", [round(v, 6) for v in p_lin.tolist()])
print("predictions   :", (p_lin >= 0.5).int().tolist(), " target:", Y.view(-1).int().tolist())

## Task 2/3 — The 2–2–1 model

`2 inputs -> 2 hidden units (nonlinear) -> 1 logit`, trained with `BCEWithLogitsLoss`
(numerically stable sigmoid + binary cross-entropy) and full-batch Adam.

Only the **hidden activation** changes between runs; the task, data, loss, optimiser,
learning rate, step count and seed are held fixed.

In [ ]:
ACTIVATIONS = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}


def build(act_name, seed=SEED, zero_init=False, identical_init=False):
    """2-2-1 network. Returns nn.Sequential: [Linear(2,2), act, Linear(2,1)]."""
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(2, 2),
        ACTIVATIONS[act_name](),
        nn.Linear(2, 1),
    )
    with torch.no_grad():
        if zero_init:                       # Part C: every weight and bias = 0
            for p in model.parameters():
                p.zero_()
        elif identical_init:                # Part C variant: the two hidden units are exact twins
            model[0].weight[1] = model[0].weight[0]
            model[0].bias[1] = model[0].bias[0]
            model[2].weight[0, 1] = model[2].weight[0, 0]
    return model


def train(model, steps=STEPS, lr=LR, probe_steps=(0, 1, 2, 10, 100), verbose=False):
    """Full-batch training. Returns a record dict with losses, probes and final gradients."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    rec = {"probes": {}, "loss_curve": []}

    for step in range(steps):
        opt.zero_grad()                       # 1. clear old gradients
        logits = model(X)                     # 2. forward pass
        loss = lossf(logits, Y)               # 3. scalar loss
        loss.backward()                       # 4. reverse-mode AD fills .grad
        if step == 0:
            rec["initial_loss"] = loss.item()
        if step in probe_steps:               # snapshot BEFORE the parameter update
            rec["probes"][step] = {
                "loss": loss.item(),
                "W1": model[0].weight.detach().clone(),
                "gW1": model[0].weight.grad.detach().clone(),
                "gW1_norm": model[0].weight.grad.norm().item(),
                "row_gap": (model[0].weight[0] - model[0].weight[1]).abs().max().item(),
            }
        rec["loss_curve"].append(loss.item())
        opt.step()                            # 5. optimiser updates parameters

    # one final forward + backward at the trained parameters, for the reported gradient tensor
    opt.zero_grad()
    logits = model(X)
    final_loss = lossf(logits, Y)
    final_loss.backward()

    probs = torch.sigmoid(logits).detach().view(-1)
    preds = (probs >= 0.5).int()
    rec.update(
        final_loss=final_loss.item(),
        probs=probs,
        preds=preds,
        all_correct=bool((preds == Y.view(-1).int()).all()),
        gW1_final=model[0].weight.grad.detach().clone(),
        gb1_final=model[0].bias.grad.detach().clone(),
        gW2_final=model[2].weight.grad.detach().clone(),
        hidden=model[:2](X).detach(),
        W1=model[0].weight.detach().clone(),
        b1=model[0].bias.detach().clone(),
        W2=model[2].weight.detach().clone(),
        b2=model[2].bias.detach().clone(),
    )
    if verbose:
        print(f"initial loss {rec['initial_loss']:.6f} -> final loss {rec['final_loss']:.6f}")
    return rec

## Task 4A — Basic learning check (main run, sigmoid hidden units)

In [ ]:
main = train(build("sigmoid"), verbose=True)

print("\ninitial loss :", f"{main['initial_loss']:.6f}")
print("final loss   :", f"{main['final_loss']:.6f}")
print("\n x1 x2 | target |  P(y=1)   | predicted")
for (x1, x2), t, p, q in zip(X.int().tolist(), Y.view(-1).int().tolist(),
                             main["probs"].tolist(), main["preds"].tolist()):
    print(f"  {x1}  {x2} |   {t}    | {p:8.6f}  |    {q}")
print("\nall four correct:", main["all_correct"])

# the learned hidden representation (why it works)
print("\nhidden activations h(x) after training:")
print(main["hidden"])
print("W1 =\n", main["W1"], "\nb1 =", main["b1"])
print("W2 =", main["W2"], " b2 =", main["b2"])

## Task 4B — Backpropagation check

`model[0].weight.grad` is $\partial L/\partial W^{(1)}$: entry $(j,i)$ is the derivative of the
**scalar** loss with respect to $W^{(1)}_{ji}$, same shape as the weight itself (2x2).
Because the loss is a *mean* over the four examples,
$L = \frac14\sum_n L_n \Rightarrow \partial L/\partial W = \frac14\sum_n \partial L_n/\partial W$,
i.e. the average of the per-example gradients. The cell verifies that identity numerically.

In [ ]:
print("dL/dW1 at the trained parameters:\n", main["gW1_final"])
print("dL/db1:", main["gb1_final"])
print("dL/dW2:", main["gW2_final"])
print("||dL/dW1||_2 =", f"{main['gW1_final'].norm().item():.3e}  (small: the model has converged)")

# --- averaging check: per-example gradients, averaged, must equal the mean-loss gradient ---
m = build("sigmoid")
lossf = nn.BCEWithLogitsLoss()
m.zero_grad(); lossf(m(X), Y).backward()
g_mean = m[0].weight.grad.clone()

acc = torch.zeros_like(g_mean)
for n in range(4):
    m.zero_grad()
    nn.BCEWithLogitsLoss()(m(X[n:n + 1]), Y[n:n + 1]).backward()
    acc += m[0].weight.grad
acc /= 4
print("\nmean-loss gradient      :\n", g_mean)
print("average of per-example  :\n", acc)
print("max |difference| =", f"{(g_mean - acc).abs().max().item():.3e}")
assert torch.allclose(g_mean, acc, atol=1e-6)

# --- independent finite-difference check of one entry (is the AD gradient the real derivative?) ---
eps = 1e-4
m2 = build("sigmoid")
with torch.no_grad():
    base = m2[0].weight[0, 0].item()
    m2[0].weight[0, 0] = base + eps
    lp = nn.BCEWithLogitsLoss()(m2(X), Y).item()
    m2[0].weight[0, 0] = base - eps
    lm = nn.BCEWithLogitsLoss()(m2(X), Y).item()
    m2[0].weight[0, 0] = base
print(f"\nfinite difference dL/dW1[0,0] = {(lp - lm) / (2 * eps):.8f}")
print(f"autograd        dL/dW1[0,0] = {g_mean[0, 0].item():.8f}")

## Task 4C — Symmetry experiment

Two variants, architecture unchanged:
1. **all-zero init** — every weight and bias set to 0;
2. **identical (non-zero) init** — the two hidden units are exact twins
   (equal rows of $W^{(1)}$, equal biases, equal outgoing weights).

Variant 2 is the sharper test: in variant 1 the first-layer gradient is identically zero
(because $\partial L/\partial h = W^{(2)\top}\delta = 0$), so nothing moves at all.

In [ ]:
zero = train(build("sigmoid", zero_init=True), probe_steps=(0, 1, 2, 10, 100, 1000))
print("=== all-zero initialisation ===")
for s, pr in zero["probes"].items():
    print(f"step {s:>4}  loss {pr['loss']:.6f}  W1 rows {pr['W1'].tolist()}  ||dL/dW1|| {pr['gW1_norm']:.3e}")
print("final loss:", f"{zero['final_loss']:.6f}", " (ln 2 =", f"{np.log(2):.6f})")
print("probabilities:", [round(v, 6) for v in zero["probs"].tolist()])
print("rows of W1 still identical:", torch.equal(zero["W1"][0], zero["W1"][1]))

ident = train(build("sigmoid", identical_init=True), probe_steps=(0, 1, 2, 10, 100, 3999))
print("\n=== identical (non-zero) initialisation ===")
for s, pr in ident["probes"].items():
    print(f"step {s:>4}  loss {pr['loss']:.6f}  max|row0-row1| = {pr['row_gap']:.3e}")
print("final loss:", f"{ident['final_loss']:.6f}")
print("probabilities:", [round(v, 6) for v in ident["probs"].tolist()])
print("predictions :", ident["preds"].tolist(), " correct:", ident["all_correct"])
print("W1 =\n", ident["W1"])
print("rows of W1 still identical:", torch.equal(ident["W1"][0], ident["W1"][1]))

## Task 4D — Activation experiment (sigmoid / tanh / ReLU)

Only the hidden activation changes. `early ||dL/dW1||` is measured at training **step 10**.
A single seed is not evidence, so the last column repeats each run over 12 seeds (0–11)
and counts how many reach 4/4.

In [ ]:
runs = {a: train(build(a), probe_steps=(0, 1, 2, 10, 100)) for a in ACTIVATIONS}

print(f"{'hidden act':<11}{'init loss':>11}{'final loss':>13}{'4/4?':>7}{'early ||gW1||':>15}")
for a, r in runs.items():
    print(f"{a:<11}{r['initial_loss']:>11.4f}{r['final_loss']:>13.6f}"
          f"{str(r['all_correct']):>7}{r['probes'][10]['gW1_norm']:>15.6f}")

print("\ngradient tensor dL/dW1 at step 10:")
for a, r in runs.items():
    print(f"  {a:<8}", r["probes"][10]["gW1"].tolist())

print("\npre-activations a1 and activations h after training (saturation / dead-unit diagnosis):")
for a, r in runs.items():
    with torch.no_grad():
        pre = X @ r["W1"].T + r["b1"]
    print(f"  {a:<8} a1 range [{pre.min():.3f}, {pre.max():.3f}]   h =\n{r['hidden']}")

# robustness over seeds
print("\nseeds 0-11, count reaching 4/4 correct:")
for a in ACTIVATIONS:
    ok = sum(train(build(a, seed=s), probe_steps=())["all_correct"] for s in range(12))
    print(f"  {a:<8} {ok}/12")

## Task 5 — Three-class extension

Same inputs, new labels: class 0 = both inactive (0,0); class 1 = disagreement (0,1),(1,0);
class 2 = both active (1,1).

Changes: the output layer becomes `Linear(2, 3)` (weight shape **3 x 2**, bias 3),
so each example produces **3 logits**; the loss becomes `nn.CrossEntropyLoss`
(log-softmax + negative log-likelihood) with integer class targets. The hidden layer is unchanged.

In [ ]:
Y3 = torch.tensor([0, 1, 1, 2])                 # class indices, dtype long

torch.manual_seed(SEED)
model3 = nn.Sequential(
    nn.Linear(2, 2),
    nn.Tanh(),
    nn.Linear(2, 3),                            # 3 logits per example
)
opt3 = torch.optim.Adam(model3.parameters(), lr=LR)
lossf3 = nn.CrossEntropyLoss()

for step in range(STEPS):
    opt3.zero_grad()
    logits3 = model3(X)                         # shape (4, 3)
    loss3 = lossf3(logits3, Y3)
    loss3.backward()
    if step == 0:
        init_loss3 = loss3.item()
    opt3.step()

with torch.no_grad():
    logits3 = model3(X)
    P = torch.softmax(logits3, dim=1)

print("output weight matrix shape:", tuple(model3[2].weight.shape), " bias:", tuple(model3[2].bias.shape))
print("logits shape:", tuple(logits3.shape), "-> 3 logits per example")
print(f"initial loss {init_loss3:.6f} -> final loss {lossf3(logits3, Y3).item():.6f}\n")

print(" x1 x2 | true |   p(class 0)  p(class 1)  p(class 2) | pred |   sum")
for (x1, x2), t, row in zip(X.int().tolist(), Y3.tolist(), P):
    print(f"  {x1}  {x2} |  {t}   |  {row[0]:.6f}    {row[1]:.6f}    {row[2]:.6f}  |  "
          f"{int(row.argmax())}   | {row.sum():.10f}")
print("\nall four correct:", bool((P.argmax(1) == Y3).all()))

### Softmax verification and the shift-invariance diagnostic

In [ ]:
row = P[1]
print("probability vector for x = (0,1):", row.tolist())
print("sum =", f"{row.sum().item():.12f}", " |sum - 1| =", f"{abs(row.sum().item() - 1):.2e}")
assert torch.allclose(P.sum(dim=1), torch.ones(4), atol=1e-6)

shifted = torch.softmax(logits3 + 100.0, dim=1)     # add the same constant to all three logits
print("\nsoftmax(z + 100) - softmax(z), max |difference| =",
      f"{(shifted - P).abs().max().item():.3e}  (invariant up to roundoff)")

# p - y : the gradient of the cross-entropy with respect to the logits
logits_g = logits3.clone().requires_grad_(True)
nn.CrossEntropyLoss(reduction="sum")(logits_g, Y3).backward()
onehot = torch.nn.functional.one_hot(Y3, 3).float()
print("\nautograd dL/dz  :\n", logits_g.grad)
print("p - y (analytic):\n", P - onehot)
print("max |difference| =", f"{(logits_g.grad - (P - onehot)).abs().max().item():.3e}")

## Report block

Prints exactly the numbers quoted in `report.pdf`, so the report can be checked against a re-run.

In [ ]:
print("=" * 62)
print(f"SEED={SEED}  STEPS={STEPS}  LR={LR}  optimiser=Adam  loss=BCEWithLogits")
print("=" * 62)
print(f"[4A] initial loss {main['initial_loss']:.6f} -> final {main['final_loss']:.6f}")
print("[4A] probabilities", [round(v, 6) for v in main["probs"].tolist()],
      "predictions", main["preds"].tolist(), "4/4:", main["all_correct"])
print("[4B] dL/dW1 =", main["gW1_final"].tolist())
print(f"[4C] zero-init  final loss {zero['final_loss']:.6f}  probs "
      f"{[round(v, 4) for v in zero['probs'].tolist()]}  rows identical "
      f"{torch.equal(zero['W1'][0], zero['W1'][1])}")
print(f"[4C] twin-init  final loss {ident['final_loss']:.6f}  probs "
      f"{[round(v, 4) for v in ident['probs'].tolist()]}  rows identical "
      f"{torch.equal(ident['W1'][0], ident['W1'][1])}")
for a, r in runs.items():
    print(f"[4D] {a:<8} final {r['final_loss']:.6f}  4/4 {str(r['all_correct']):<5} "
          f"early ||gW1|| {r['probes'][10]['gW1_norm']:.6f}")
print(f"[5 ] final CE loss {lossf3(logits3, Y3).item():.6f}  predictions {P.argmax(1).tolist()}")
for i, r in enumerate(P):
    print(f"[5 ] x={X[i].int().tolist()}  p = {[round(v, 6) for v in r.tolist()]}  sum={r.sum():.10f}")